## Download Packages and Clone Github Repo for Data

In [ ]:
!pip install torch transformers datasets
!git clone https://github.com/google-research/distilling-step-by-step.git
%cd distilling-step-by-step
!unzip datasets.zip

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 480.6/480.6 kB 10.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 10.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 179.3/179.3 kB 15.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 134.8/134.8 kB 12.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.1/194.1 kB 12.1 MB/s eta 0:00:00
  Attempting uninstall: fsspec
    Found existing installation: fsspec 2024.10.0
    Uninstalling fsspec-2024.10.0:
      Successfully uninstalled fsspec-2024.10.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gcsfs 2024.10.0 requires fsspec==2024.10.0, but you have fsspec 2024.9.0 which is incompatible.
Cloning into 'distilling-step-by-step'...
remote: Enumerating objects: 41, done.
remote: Counting objects: 100% (22/22), done.
remote: Compressing objects: 100%

# Load Packages

In [ ]:
from data_utils import CQADatasetLoader

from transformers import T5ForConditionalGeneration, T5Tokenizer
from torch.utils.data import DataLoader
from transformers import AdamW
import torch
from tqdm import tqdm
import random

from sklearn.metrics import accuracy_score

# Load Model and Tokenizer

In [ ]:
# load the pre trained model T5 and tokenizer
model_name = "google/t5-v1_1-small"
tokenizer = T5Tokenizer.from_pretrained(model_name)

# special tokens for rationale and label handling
special_tokens = {"additional_special_tokens": ["[label]", "[rationale]"]}
tokenizer.add_special_tokens(special_tokens)



model = T5ForConditionalGeneration.from_pretrained(model_name)
device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)
print("Model and tokenizer loaded.")

/usr/local/lib/python3.10/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/1.86k [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/1.79k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/537 [00:00<?, ?B/s]

You are using the default legacy behaviour of the <class 'transformers.models.t5.tokenization_t5.T5Tokenizer'>. This is expected, and simply means that the `legacy` (previous) behavior will be used so nothing changes for you. If you want to use the new behaviour, set `legacy=False`. This should only be set if you understand what it means, and thoroughly read the reason why this was added as explained in https://github.com/huggingface/transformers/pull/24565


pytorch_model.bin:   0%|          | 0.00/308M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

Model and tokenizer loaded.


# Load Raw Dataset

In [ ]:
cqa_loader = CQADatasetLoader()
processed_data = cqa_loader.load_from_json()

train_rationales, train_labels = cqa_loader.load_llm_preds('train')
test_rationales, test_labels = cqa_loader.load_llm_preds('test')

print("Sample Processed Train Data:", processed_data['train'][0])
print("Sample Train Rationale:", train_rationales[0])
print("Sample Train Label:", train_labels[0])

Generating train split: 0 examples [00:00, ? examples/s]

Generating test split: 0 examples [00:00, ? examples/s]

Map:   0%|          | 0/9741 [00:00<?, ? examples/s]

Map:   0%|          | 0/1221 [00:00<?, ? examples/s]

Sample Processed Train Data: {'input': '"There are 10 apples on an apple tree.  Three fall off.  Now there are X apples."  What is this an example of?\nAnswer Choices:\n(a) park\n(b) coloring book\n(c) garden center\n(d) math problem\n(e) gravity', 'label': 'math problem'}
Sample Train Rationale: The answer must be something that has to do with the above problem. Of the above choices, only math problem has to do with the above problem.
Sample Train Label: math problem


# Tokenize Data and Create Dataloader for Standard Distillation

In [ ]:
max_len = 64

In [ ]:
# Prepare label-only datasets
train_label_data = []
for example, label in zip(processed_data['train'], train_labels):
    input_label = f"{example['input']}"
    train_label_data.append({"input": input_label, "label": label})

test_label_data = []
for example, label in zip(processed_data['test'], test_labels):
    input_label = f"{example['input']}"
    test_label_data.append({"input": input_label, "label": label})

In [ ]:
# tokenize training data for standard distillation
# each input is padded or truncated to a fixed length and converted to tensors

tokenized_train_label = []
for example in train_label_data:

    # tokenize inputs
    inputs = tokenizer(
        example['input'],
        truncation=True,
        padding="max_length",
        max_length=max_len,
        return_tensors="pt"
    )

    # tokenize the corresponding labels
    labels = tokenizer(
        example['label'],
        truncation=True,
        padding="max_length",
        max_length=max_len,
        return_tensors="pt"
    )


    tokenized_train_label.append({
        "input_ids": inputs['input_ids'].squeeze(),
        "attention_mask": inputs['attention_mask'].squeeze(),
        "labels": labels['input_ids'].squeeze()
    })

# tokenize test data in a similar manner for evaluation
tokenized_test_label = []
for example in test_label_data:
    inputs = tokenizer(
        example['input'],
        truncation=True,
        padding="max_length",
        max_length=max_len,
        return_tensors="pt"
    )
    labels = tokenizer(
        example['label'],
        truncation=True,
        padding="max_length",
        max_length=max_len,
        return_tensors="pt"
    )
    tokenized_test_label.append({
        "input_ids": inputs['input_ids'].squeeze(),
        "attention_mask": inputs['attention_mask'].squeeze(),
        "labels": labels['input_ids'].squeeze()
    })

In [ ]:
# few samples and tokenized versions of input and labels
print("Sample Tokenized Train Label:", tokenized_train_label[0])
print("Sample Tokenized Test Label:", tokenized_test_label[0])

Sample Tokenized Train Label: {'input_ids': tensor([   96,  7238,    33,   335, 16981,    30,    46,  8947,  2195,     5,
         5245,  1590,   326,     5,   852,   132,    33,     3,     4, 16981,
          535,   363,    19,    48,    46,   677,    13,    58, 11801, 13745,
            7,    10,    41,     9,    61,  2447,    41,   115,    61,  7089,
          484,    41,    75,    61,  2004,  1530,    41,    26,    61,  7270,
          682,    41,    15,    61, 18076,     1,     0,     0,     0,     0,
            0,     0,     0,     0]), 'attention_mask': tensor([1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
        1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
        1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0]), 'labels': tensor([7270,  682,    1,    0,    0,    0,    0,    0,    0,    0,    0,    0,
           0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
           0,    0,    0,    0,    0,    0, 

## Defining the training parameters and initializing the optimizer

In [ ]:
train_loader_label = DataLoader(tokenized_train_label, batch_size=batch_size, shuffle=True)
val_loader_label = DataLoader(tokenized_test_label, batch_size=batch_size)

In [ ]:
batch_size = 32
num_epochs = 5
learning_rate = 5e-4
optimizer = AdamW(model.parameters(), lr=learning_rate)

/usr/local/lib/python3.10/dist-packages/transformers/optimization.py:591: FutureWarning: This implementation of AdamW is deprecated and will be removed in a future version. Use the PyTorch implementation torch.optim.AdamW instead, or set `no_deprecation_warning=True` to disable this warning
  warnings.warn(


# Evaluate Original model

In [ ]:
model.eval()
predictions, references = [], []

with torch.no_grad():
    for batch in tqdm(val_loader_label, desc="Evaluating"):
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)

        outputs = model.generate(input_ids=input_ids, attention_mask=attention_mask, max_length=max_len)
        preds = [tokenizer.decode(output, skip_special_tokens=True) for output in outputs]
        refs = [tokenizer.decode(label, skip_special_tokens=True) for label in batch['labels']]

        predictions.extend(preds)
        references.extend(refs)

print(f"\nAccuracy Before Standard Distillation: {accuracy_score(references, predictions)*100:.4f}%")

Evaluating: 100%|██████████| 39/39 [00:41<00:00,  1.05s/it]


Accuracy Before Standard Distillation: 0.0000%


# Standard Distillation

In [ ]:
# set the model to training mode
model.train()
for epoch in range(num_epochs):
    print(f"Epoch {epoch + 1}/{num_epochs}")
    total_loss = 0
    for batch in tqdm(train_loader_label, desc="Training"):
        optimizer.zero_grad()

        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)

        outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
        loss = outputs.loss

        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    print(f"Epoch Loss: {total_loss / len(train_loader_label)}")

Epoch 1/5


Training: 100%|██████████| 305/305 [01:35<00:00,  3.20it/s]


Epoch Loss: 3.0370953010731054
Epoch 2/5


Training: 100%|██████████| 305/305 [01:35<00:00,  3.21it/s]


Epoch Loss: 0.4138484532715844
Epoch 3/5


...

Epoch Loss: 0.2890118721078654
Epoch 4/5


Training: 100%|██████████| 305/305 [01:35<00:00,  3.21it/s]


Epoch Loss: 0.19948355648361268
Epoch 5/5


Training: 100%|██████████| 305/305 [01:35<00:00,  3.21it/s]

Epoch Loss: 0.1655306743549519


# Evaluate Model After Standard Distillation

In [ ]:
model.eval()
predictions, references = [], []
with torch.no_grad():
    for batch in tqdm(val_loader_label, desc="Evaluating"):
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)

        outputs = model.generate(input_ids=input_ids, attention_mask=attention_mask, max_length=max_len)
        preds = [tokenizer.decode(output, skip_special_tokens=True) for output in outputs]
        refs = [tokenizer.decode(label, skip_special_tokens=True) for label in batch['labels']]

        predictions.extend(preds)
        references.extend(refs)

print(f"\nAccuracy After Standard Distillation: {accuracy_score(references, predictions)*100:.4f}%")

Evaluating: 100%|██████████| 39/39 [00:46<00:00,  1.19s/it]


Accuracy After Standard Distillation: 14.0868%


# Prepare Data for Step-by-Step Distillation


In [ ]:
fraction = 0.5
random_seed = 42

In [ ]:
train_multitask_data = []
random.seed(random_seed)
sample_size = int(len(processed_data['train']) * fraction)
sampled_indices = random.sample(range(len(processed_data['train'])), sample_size)

for i in sampled_indices:
    example = processed_data['train'][i]
    rationale = train_rationales[i]
    label = train_labels[i]
    # add rationale and label prefixes to create distinct input formats
    input_rationale = f"[rationale] {example['input']}"
    input_label = f"[label] {example['input']}"
    train_multitask_data.append({
        "input_rationale": input_rationale,
        "input_label": input_label,
        "label": label,
        "rationale": rationale
    })


test_multitask_data = []
for example, label in zip(processed_data['test'], test_labels):
    input_label = f"[label] {example['input']}"
    test_multitask_data.append({
        "input": input_label,
        "label": label
    })

In [ ]:
# Tokenize datasets
tokenized_train_multitask = []
for example in train_multitask_data:
    inputs_rationale = tokenizer(
        example['input_rationale'],
        truncation=True,
        padding="max_length",
        max_length=max_len,
        return_tensors="pt"
    )
    inputs_label = tokenizer(
        example['input_label'],
        truncation=True,
        padding="max_length",
        max_length=max_len,
        return_tensors="pt"
    )
    labels = tokenizer(
        example['label'],
        truncation=True,
        padding="max_length",
        max_length=max_len,
        return_tensors="pt"
    )
    rationales = tokenizer(
        example['rationale'],
        truncation=True,
        padding="max_length",
        max_length=max_len,
        return_tensors="pt"
    )
    tokenized_train_multitask.append({
        "input_ids_rationale": inputs_rationale['input_ids'].squeeze(),
        "attention_mask_rationale": inputs_rationale['attention_mask'].squeeze(),
        "input_ids_label": inputs_label['input_ids'].squeeze(),
        "attention_mask_label": inputs_label['attention_mask'].squeeze(),
        "labels": labels['input_ids'].squeeze(),
        "rationales": rationales['input_ids'].squeeze()
    })

tokenized_test_multitask = []
for example in test_multitask_data:
    inputs = tokenizer(
        example['input'],
        truncation=True,
        padding="max_length",
        max_length=max_len,
        return_tensors="pt"
    )
    labels = tokenizer(
        example['label'],
        truncation=True,
        padding="max_length",
        max_length=max_len,
        return_tensors="pt"
    )
    tokenized_test_multitask.append({
        "input_ids": inputs['input_ids'].squeeze(),
        "attention_mask": inputs['attention_mask'].squeeze(),
        "labels": labels['input_ids'].squeeze()
    })

# Reload Model

In [ ]:
model = T5ForConditionalGeneration.from_pretrained("google/t5-v1_1-small")
model.resize_token_embeddings(len(tokenizer))
model.to(device)
print("Model and tokenizer loaded.")

Model and tokenizer loaded.


# Create Data Loaders

In [ ]:
train_loader_multitask = DataLoader(tokenized_train_multitask, batch_size=batch_size, shuffle=True)
test_loader_multitask = DataLoader(tokenized_test_multitask, batch_size=batch_size)

optimizer = AdamW(model.parameters(), lr=learning_rate)
lambda_value = 0.2

/usr/local/lib/python3.10/dist-packages/transformers/optimization.py:591: FutureWarning: This implementation of AdamW is deprecated and will be removed in a future version. Use the PyTorch implementation torch.optim.AdamW instead, or set `no_deprecation_warning=True` to disable this warning
  warnings.warn(


# Step By Step Distillation Training

In [ ]:
model.train()
lambda_value = 0.5

for phase in ["rationale", "label"]:
    print(f"Starting training phase: {phase.upper()}")

    for epoch in range(num_epochs):
        print(f"Epoch {epoch + 1}/{num_epochs} ({phase.upper()})")
        total_loss = 0
        total_label_loss = 0
        total_rationale_loss = 0

        for batch in tqdm(train_loader_multitask, desc=f"Training {phase}"):
            optimizer.zero_grad()

            if phase == "rationale":
                input_ids = batch['input_ids_rationale'].to(device)
                attention_mask = batch['attention_mask_rationale'].to(device)
                targets = batch['rationales'].to(device)

                outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=targets)
                loss = lambda_value * outputs.loss  # Scale rationale loss
                total_rationale_loss += loss.item()
            elif phase == "label":
                input_ids = batch['input_ids_label'].to(device)
                attention_mask = batch['attention_mask_label'].to(device)
                targets = batch['labels'].to(device)

                outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=targets)
                loss = outputs.loss
                total_label_loss += loss.item()

            loss.backward()
            optimizer.step()

            total_loss += loss.item()

        avg_total_loss = total_loss / len(train_loader_multitask)
        avg_rationale_loss = total_rationale_loss / len(train_loader_multitask) if phase == "rationale" else 0
        avg_label_loss = total_label_loss / len(train_loader_multitask) if phase == "label" else 0

        print(f"Epoch {epoch + 1} Loss ({phase.upper()}): {avg_total_loss:.4f}")
        if phase == "rationale":
            print(f"  Avg Rationale Loss: {avg_rationale_loss:.4f}")
        if phase == "label":
            print(f"  Avg Label Loss: {avg_label_loss:.4f}")

Starting training phase: RATIONALE
Epoch 1/5 (RATIONALE)


Training rationale: 100%|██████████| 153/153 [00:49<00:00,  3.08it/s]


Epoch 1 Loss (RATIONALE): 2.4375
  Avg Rationale Loss: 2.4375
Epoch 2/5 (RATIONALE)


Training rationale: 100%|██████████| 153/153 [00:47<00:00,  3.19it/s]


Epoch 2 Loss (RATIONALE): 0.4024
  Avg Rationale Loss: 0.4024
Epoch 3/5 (RATIONALE)


...

Epoch 3 Loss (LABEL): 0.2095
  Avg Label Loss: 0.2095
Epoch 4/5 (LABEL)


Training label: 100%|██████████| 153/153 [00:48<00:00,  3.18it/s]


Epoch 4 Loss (LABEL): 0.1549
  Avg Label Loss: 0.1549
Epoch 5/5 (LABEL)


Training label: 100%|██████████| 153/153 [00:48<00:00,  3.17it/s]

Epoch 5 Loss (LABEL): 0.1266
  Avg Label Loss: 0.1266


# Evaluate Model After Step by Step Distillation

In [ ]:
model.eval()
predictions, references = [], []
with torch.no_grad():
    for batch in tqdm(test_loader_multitask, desc="Evaluating"):
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)

        outputs = model.generate(input_ids=input_ids, attention_mask=attention_mask, max_length=max_len)
        preds = [tokenizer.decode(output, skip_special_tokens=True) for output in outputs]
        refs = [tokenizer.decode(label, skip_special_tokens=True) for label in batch['labels']]

        predictions.extend(preds)
        references.extend(refs)

print(f"\nAccuracy After Multitask Distillation: {accuracy_score(references, predictions)*100:.4f}%")

Evaluating: 100%|██████████| 39/39 [00:17<00:00,  2.28it/s]


Accuracy After Multitask Distillation: 25.4709%
